#My temporary idea: pipeline overview

1. Data understanding, EDA
2. Test relationships and construct relationshp graph
3. Features engenieering, turn graph into ML input
4. ML model
5. Cross wafer validation
6. Evaluation and interpretation

First we're loading up the data and organizing the files in 4 csv:
- 1 csv with the threshold (only two rows)
- 3 csv, one for each wafer

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.metrics import jaccard_score
from sklearn.cluster import AgglomerativeClustering

print(" STEP 1 — EDA STARTING ")

# 0. LOAD DATA
print("Loading dataset...")

df = pd.read_csv("data.csv")

print(f"Dataset shape: {df.shape}")
print("First rows preview:")
print(df.head(), "\n")

In [ ]:
print("All column names:")
for i, col in enumerate(df.columns, 1):
    print(f"{i:3}. {col}")

# Also show count
print(f"Total columns: {len(df.columns)}")

In [ ]:
# 1. IDENTIFY WAFERS

print("Splitting by wafer...")

if "Wafer" in df.columns:
    wafers = df["Wafer"].unique()
    wafer_groups = {w: df[df["Wafer"] == w].copy() for w in wafers}
else:
    raise ValueError("No 'Wafer' column found. Please check dataset structure.")

print(f"Found wafers: {list(wafers)}")

In [ ]:
# 1. EXTRACT THRESHOLD LINES
print("Extracting threshold lines...")

# Extract first 2 lines as thresholds
threshold_lines = df.iloc[:2].copy()
threshold_lines.to_csv("thresholds.csv", index=False)
print(f"Saved first 2 lines (thresholds) to 'thresholds.csv'")
print(f"Threshold lines shape: {threshold_lines.shape}\n")

# Remove threshold lines from main dataframe
df_data = df.iloc[2:].reset_index(drop=True)
print(f"Data after removing thresholds: {df_data.shape}")


In [ ]:
# SIMPLIFIED EXTRACTION & GROUPING

print("Extracting and grouping by wafer number...")

# Extract wafer number using simple string slicing
# The pattern is always 'DMKYXXX-' where XXX is the wafer number
df_data['Wafer_Number'] = df_data['Wafer'].str.extract(r'DMKY(\d{3})-')

# Remove rows without valid wafer number (should be none if all follow pattern)
df_data_clean = df_data.dropna(subset=['Wafer_Number']).copy()

# Group by wafer number and save in one go
for wafer_num, group in df_data_clean.groupby('Wafer_Number'):
    filename = f"wafer_{wafer_num}.csv"
    group.drop('Wafer_Number', axis=1).to_csv(filename, index=False)
    print(f"  ✓ Wafer {wafer_num}: {len(group)} rows → '{filename}'")

print(f"\n✓ Created files for wafers: {df_data_clean['Wafer_Number'].unique().tolist()}")

In [ ]:
# Check shapes of all wafer CSV files
for wafer_num in ['801', '806', '812']:
    filename = f"wafer_{wafer_num}.csv"
    try:
        df = pd.read_csv(filename)
        print(f"{filename}: {df.shape[0]} rows, {df.shape[1]} columns")
    except FileNotFoundError:
        print(f"{filename}: NOT FOUND")

In [ ]:
print("Separating test columns from metadata...")

# Define metadata columns (based on what you saw)
meta_cols = [
    "Source Lot", "Lot", "Wafer", "rework_flag", "Program",
    "temperature", "subid", "site", "die_x", "die_y",
    "device_nr", "rom_code", "hardbin", "lib_info",
    "BinName", "BinState"
]

# Keep only columns that exist (safety)
meta_cols = [c for c in meta_cols if c in df_data_clean.columns]

# Extract test columns
test_cols = [c for c in df_data_clean.columns if c not in meta_cols]

print(f"Metadata columns: {len(meta_cols)}")
print(f"Test columns: {len(test_cols)}")

# Create clean test-only dataset
df_tests = df_data_clean[test_cols].copy()

print("Test-only dataset shape:", df_tests.shape)

In [ ]:
print("Cleaning numeric test data...")

# Convert everything to numeric
df_tests = df_tests.apply(pd.to_numeric, errors='coerce')

# Check missing values
missing_ratio = df_tests.isna().mean()

print("Average missing ratio:", missing_ratio.mean())

# Drop bad columns (too many NaNs)
threshold = 0.2  # you can tune this
good_cols = missing_ratio[missing_ratio < threshold].index

df_tests = df_tests[good_cols]

print(f"Remaining test columns after cleaning: {df_tests.shape[1]}")

Now we compute the convergence matrix: Think of each test as a “sensor” watching thousands of chips: the correlation matrix is just checking whether two sensors wiggle together across all those chips—if whenever test A is a bit high on a chip, test B is also a bit high, they get a strong positive correlation (close to 1); if one goes up while the other goes down, it’s negative; and if they behave independently, it’s near zero—so you’re not measuring failure yet, just whether tests behave like copies, cousins, or strangers across the same population.

In [ ]:
print("\nComputing correlation matrix...")

corr_matrix = df_tests.corr(method='pearson')

print("Correlation matrix shape:", corr_matrix.shape)

In [ ]:
plt.figure(figsize=(12, 10))
plt.imshow(corr_matrix, aspect='auto', vmin=-1, vmax=1)
plt.colorbar()
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()